# 06 — Statistical Analysis

**Objetivo:** Validar com rigor estatístico se os padrões observados no EDA são reais ou produto do acaso.

**Testes aplicados:**
1. **Chi-quadrado** — taxa de falha difere significativamente entre regiões?
2. **Chi-quadrado** — taxa de falha difere significativamente entre dias da semana?
3. **Kruskal-Wallis** — valor dos pedidos difere entre regiões?
4. **Mann-Whitney U** — motoristas de alto risco vs. baixo risco: diferença real?
5. **Intervalos de confiança (95%)** — para a taxa de falha por região
6. **Effect size (Cramér's V)** — magnitude real das diferenças encontradas

> Um analista pleno não para no gráfico — ele prova estatisticamente que o padrão é real.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, norm
import warnings
warnings.filterwarnings('ignore')

master = pd.read_parquet('../data/processed/master.parquet')
sns.set_theme(style='whitegrid', palette='Blues_d')
print(f'Pedidos carregados: {master.shape[0]:,}')
print(f'Taxa global de falha: {master["has_missing"].mean()*100:.1f}%')

## 1. Chi-quadrado — Taxa de falha por Região

**H₀:** A taxa de itens faltando é igual em todas as regiões  
**H₁:** Pelo menos uma região tem taxa significativamente diferente

In [ ]:
# Tabela de contingência: região × (com falha / sem falha)
contingency_region = pd.crosstab(master['region'], master['has_missing'])

chi2_r, p_r, dof_r, expected_r = chi2_contingency(contingency_region)

# Cramér's V — effect size
n = contingency_region.sum().sum()
k = min(contingency_region.shape) - 1
cramers_v_r = np.sqrt(chi2_r / (n * k))

print('=' * 55)
print('  TESTE CHI-QUADRADO — REGIÃO × FALHA')
print('=' * 55)
print(f'  Chi²:        {chi2_r:.4f}')
print(f'  p-value:     {p_r:.6f}')
print(f'  Graus lib.:  {dof_r}')
print(f"  Cramér's V:  {cramers_v_r:.4f}")
print()

if p_r < 0.05:
    print('  RESULTADO: Diferença SIGNIFICATIVA entre regiões (p < 0.05)')
    if cramers_v_r < 0.1:
        print(f"  MAGNITUDE: Pequena (Cramér's V = {cramers_v_r:.3f})")
    elif cramers_v_r < 0.3:
        print(f"  MAGNITUDE: Moderada (Cramér's V = {cramers_v_r:.3f})")
    else:
        print(f"  MAGNITUDE: Grande (Cramér's V = {cramers_v_r:.3f})")
else:
    print('  RESULTADO: Sem diferença significativa entre regiões (p >= 0.05)')

## 2. Intervalos de Confiança 95% por Região

Intervalo de Wilson para proporções — mais robusto que o intervalo normal para taxas.

In [ ]:
def wilson_ci(successes, n, confidence=0.95):
    """Intervalo de confiança de Wilson para proporções."""
    z = norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denominator
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denominator
    return center - margin, center + margin

region_stats = (
    master.groupby('region')
    .agg(total=('order_id', 'count'), missing=('has_missing', 'sum'))
    .reset_index()
)
region_stats['rate'] = region_stats['missing'] / region_stats['total']
ci = region_stats.apply(
    lambda r: wilson_ci(r['missing'], r['total']), axis=1, result_type='expand'
)
region_stats['ci_low'] = ci[0]
region_stats['ci_high'] = ci[1]
region_stats['error_low']  = region_stats['rate'] - region_stats['ci_low']
region_stats['error_high'] = region_stats['ci_high'] - region_stats['rate']
region_stats = region_stats.sort_values('rate', ascending=False)

global_rate = master['has_missing'].mean()

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#c0392b' if r > global_rate else '#2980b9' for r in region_stats['rate']]
ax.barh(
    region_stats['region'],
    region_stats['rate'] * 100,
    xerr=[region_stats['error_low'] * 100, region_stats['error_high'] * 100],
    color=colors, capsize=5, error_kw={'linewidth': 2}
)
ax.axvline(global_rate * 100, color='black', linestyle='--', linewidth=1.5,
           label=f'Média global: {global_rate*100:.1f}%')
ax.set_title('Taxa de Itens Faltando por Região\ncom Intervalos de Confiança 95% (Wilson)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Taxa de Falha (%)')
red_patch = mpatches.Patch(color='#c0392b', label='Acima da média')
blue_patch = mpatches.Patch(color='#2980b9', label='Abaixo da média')
ax.legend(handles=[red_patch, blue_patch, plt.Line2D([0],[0], color='black', linestyle='--')],
          labels=['Acima da média', 'Abaixo da média', f'Média global: {global_rate*100:.1f}%'])
plt.tight_layout()
plt.savefig('../reports/figures/13_ci_by_region.png', dpi=150)
plt.show()

print('\nTabela completa:')
display(region_stats[['region', 'total', 'missing', 'rate', 'ci_low', 'ci_high']]
        .assign(
            rate=lambda d: (d['rate'] * 100).round(1).astype(str) + '%',
            ci_low=lambda d: (d['ci_low'] * 100).round(1).astype(str) + '%',
            ci_high=lambda d: (d['ci_high'] * 100).round(1).astype(str) + '%'
        )
        .rename(columns={'ci_low': 'IC_baixo', 'ci_high': 'IC_alto'}))

## 3. Chi-quadrado — Taxa de falha por Dia da Semana

In [ ]:
contingency_day = pd.crosstab(master['day_of_week'], master['has_missing'])
chi2_d, p_d, dof_d, _ = chi2_contingency(contingency_day)

n_d = contingency_day.sum().sum()
k_d = min(contingency_day.shape) - 1
cramers_v_d = np.sqrt(chi2_d / (n_d * k_d))

print('=' * 55)
print('  TESTE CHI-QUADRADO — DIA DA SEMANA × FALHA')
print('=' * 55)
print(f'  Chi²:        {chi2_d:.4f}')
print(f'  p-value:     {p_d:.6f}')
print(f"  Cramér's V:  {cramers_v_d:.4f}")
print()
if p_d < 0.05:
    print('  RESULTADO: Dia da semana influencia significativamente a taxa de falha')
else:
    print('  RESULTADO: Dia da semana NÃO influencia significativamente a taxa de falha')

# Visualização com IC por dia
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = (
    master.groupby('day_of_week')
    .agg(total=('order_id', 'count'), missing=('has_missing', 'sum'))
    .reindex(day_order)
    .reset_index()
)
day_stats['rate'] = day_stats['missing'] / day_stats['total']
ci_d = day_stats.apply(lambda r: wilson_ci(r['missing'], r['total']), axis=1, result_type='expand')
day_stats['err_low']  = day_stats['rate'] - ci_d[0]
day_stats['err_high'] = ci_d[1] - day_stats['rate']

fig, ax = plt.subplots(figsize=(11, 5))
colors_d = ['#c0392b' if r > global_rate else '#2980b9' for r in day_stats['rate']]
ax.bar(day_stats['day_of_week'], day_stats['rate'] * 100,
       yerr=[day_stats['err_low'] * 100, day_stats['err_high'] * 100],
       color=colors_d, capsize=5, error_kw={'linewidth': 2})
ax.axhline(global_rate * 100, color='black', linestyle='--', linewidth=1.5,
           label=f'Média global: {global_rate*100:.1f}%')
ax.set_title(f'Taxa de Falha por Dia da Semana com IC 95%\n(Chi²={chi2_d:.2f}, p={p_d:.4f}, V={cramers_v_d:.3f})',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Taxa de Falha (%)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/14_ci_by_weekday.png', dpi=150)
plt.show()

## 4. Kruskal-Wallis — Valor do Pedido por Região

Testa se o valor médio dos pedidos é estatisticamente diferente entre regiões (alternativa não-paramétrica ao ANOVA).

In [ ]:
region_groups = [grp['order_amount'].values for _, grp in master.groupby('region')]
stat_kw, p_kw = kruskal(*region_groups)

print('=' * 55)
print('  KRUSKAL-WALLIS — VALOR DO PEDIDO POR REGIÃO')
print('=' * 55)
print(f'  H statistic: {stat_kw:.4f}')
print(f'  p-value:     {p_kw:.6f}')
print()
if p_kw < 0.05:
    print('  RESULTADO: Diferença significativa nos valores de pedido entre regiões')
else:
    print('  RESULTADO: Sem diferença significativa — regiões têm perfil de pedido similar')

# Boxplot
region_order = master.groupby('region')['order_amount'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 6))
master_sorted = master.copy()
master_sorted['region'] = pd.Categorical(master_sorted['region'], categories=region_order, ordered=True)
master_sorted_grouped = [master_sorted[master_sorted['region'] == r]['order_amount'].values for r in region_order]
bp = ax.boxplot(master_sorted_grouped, labels=region_order, patch_artist=True, notch=True)
for patch in bp['boxes']:
    patch.set_facecolor('#4a90d9')
    patch.set_alpha(0.7)
ax.set_title(f'Distribuição do Valor dos Pedidos por Região\n(Kruskal-Wallis H={stat_kw:.2f}, p={p_kw:.4f})',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Valor do Pedido ($)')
ax.set_xlabel('Região')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/15_order_amount_boxplot.png', dpi=150)
plt.show()

## 5. Mann-Whitney U — Alto risco vs. Baixo risco (Motoristas)

Compara a taxa de falha entre os 25% piores motoristas e os 25% melhores.

In [ ]:
driver_perf = (
    master.groupby('driver_id')
    .agg(deliveries=('order_id', 'count'), missing_rate=('has_missing', 'mean'))
    .reset_index()
)
driver_perf = driver_perf[driver_perf['deliveries'] >= 5]

q75 = driver_perf['missing_rate'].quantile(0.75)
q25 = driver_perf['missing_rate'].quantile(0.25)

high_risk = driver_perf[driver_perf['missing_rate'] >= q75]['missing_rate'].values
low_risk  = driver_perf[driver_perf['missing_rate'] <= q25]['missing_rate'].values

stat_mw, p_mw = mannwhitneyu(high_risk, low_risk, alternative='greater')

# Cohen's d
pooled_std = np.sqrt((np.std(high_risk)**2 + np.std(low_risk)**2) / 2)
cohens_d = (np.mean(high_risk) - np.mean(low_risk)) / pooled_std if pooled_std > 0 else 0

print('=' * 55)
print('  MANN-WHITNEY — ALTO RISCO vs. BAIXO RISCO')
print('=' * 55)
print(f'  Motoristas alto risco  (Q75+): {len(high_risk)}')
print(f'  Motoristas baixo risco (Q25-): {len(low_risk)}')
print(f'  Média alto risco:  {np.mean(high_risk)*100:.1f}%')
print(f'  Média baixo risco: {np.mean(low_risk)*100:.1f}%')
print(f'  U statistic: {stat_mw:.1f}')
print(f'  p-value:     {p_mw:.6f}')
print(f"  Cohen's d:   {cohens_d:.4f}")
print()
if p_mw < 0.05:
    print('  RESULTADO: Motoristas de alto risco têm taxa de falha SIGNIFICATIVAMENTE maior')
    if cohens_d > 0.8:
        print(f"  MAGNITUDE: Grande (d = {cohens_d:.2f}) — diferença operacionalmente relevante")
    elif cohens_d > 0.5:
        print(f"  MAGNITUDE: Moderada (d = {cohens_d:.2f})")
    else:
        print(f"  MAGNITUDE: Pequena (d = {cohens_d:.2f})")

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(high_risk * 100, bins=20, alpha=0.7, color='#c0392b', label=f'Alto risco (n={len(high_risk)})')
ax.hist(low_risk  * 100, bins=20, alpha=0.7, color='#27ae60', label=f'Baixo risco (n={len(low_risk)})')
ax.axvline(np.mean(high_risk) * 100, color='#c0392b', linestyle='--', linewidth=2)
ax.axvline(np.mean(low_risk)  * 100, color='#27ae60', linestyle='--', linewidth=2)
ax.set_title(f'Distribuição de Falhas: Alto Risco vs. Baixo Risco\n(Mann-Whitney p={p_mw:.4f}, d={cohens_d:.2f})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Taxa de Itens Faltando (%)')
ax.set_ylabel('Número de Motoristas')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/16_driver_risk_distribution.png', dpi=150)
plt.show()

## 6. Resumo dos Testes Estatísticos

In [ ]:
summary = pd.DataFrame([
    {
        'Hipótese testada': 'Região afeta taxa de falha?',
        'Teste': 'Chi-quadrado',
        'Estatística': f'{chi2_r:.2f}',
        'p-value': f'{p_r:.4f}',
        'Effect size': f"V={cramers_v_r:.3f}",
        'Conclusão': 'Sim*' if p_r < 0.05 else 'Não'
    },
    {
        'Hipótese testada': 'Dia da semana afeta taxa de falha?',
        'Teste': 'Chi-quadrado',
        'Estatística': f'{chi2_d:.2f}',
        'p-value': f'{p_d:.4f}',
        'Effect size': f"V={cramers_v_d:.3f}",
        'Conclusão': 'Sim*' if p_d < 0.05 else 'Não'
    },
    {
        'Hipótese testada': 'Valor do pedido difere entre regiões?',
        'Teste': 'Kruskal-Wallis',
        'Estatística': f'{stat_kw:.2f}',
        'p-value': f'{p_kw:.4f}',
        'Effect size': 'N/A',
        'Conclusão': 'Sim*' if p_kw < 0.05 else 'Não'
    },
    {
        'Hipótese testada': 'Alto risco > baixo risco (motoristas)?',
        'Teste': 'Mann-Whitney U',
        'Estatística': f'{stat_mw:.0f}',
        'p-value': f'{p_mw:.4f}',
        'Effect size': f"d={cohens_d:.2f}",
        'Conclusão': 'Sim*' if p_mw < 0.05 else 'Não'
    },
])

print('RESUMO DOS TESTES ESTATÍSTICOS')
print('* = significativo ao nível 0.05')
display(summary)